# SurvFace Grad-CAM — 02. Population Grad-CAM extraction

공식 registered/unmated probe에서 설정된 coverage에 따라 얼굴과 다중 부위 Grad-CAM을 계산합니다. 압축/open-set 평가는 이 표본 제한과 무관하게 전체 protocol을 사용합니다.

이 노트북은 한 단계만 실행하는 thin runbook입니다. 계산 구현은 `research/experiments/step4_workflow.py`에 있습니다.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
EXECUTION = CONFIG["execution"]
MODEL_PROFILE = str(EXECUTION["model_profile"])
MODE = str(EXECUTION["mode"])
DATA_FRACTION = float(EXECUTION["data_fraction"])
EXECUTE_STAGE = bool(EXECUTION["execute_stage"])
WRITE_OUTPUTS = bool(EXECUTION["write_outputs"])
OVERWRITE = bool(EXECUTION["overwrite"])
DATASET_ID = "survface"

if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")
if EXECUTE_STAGE and not WRITE_OUTPUTS:
    raise ValueError("정식 단계 실행은 WRITE_OUTPUTS=True여야 합니다.")

from research.experiments import extract_step4_population_gradcam
from research.runtime import ProgressReporter

PROGRESS = ProgressReporter(
    "SurvFace population Grad-CAM",
    heartbeat_seconds=None,
    milestone_percent=10,
)


In [2]:
if EXECUTE_STAGE:
    result = extract_step4_population_gradcam(
        CONFIG_PATH,
        project_root=PROJECT_ROOT,
        dataset_id=DATASET_ID,
        execution_acknowledged=True,
        progress=PROGRESS.callback(key_prefix=f"{DATASET_ID}:step4:"),
    )
else:
    result = {
        "dataset_id": DATASET_ID,
        "status": "not_executed",
        "reason": "CONFIG execution gates are closed",
    }

result


[23:03:24] SurvFace population Grad-CAM | population Grad-CAM extraction | elapsed=2m 31s | progress=10% processed=18216 total=182159 rate=120.62/s eta=22m 39s
[23:05:14] SurvFace population Grad-CAM | population Grad-CAM extraction | elapsed=4m 21s | progress=20% processed=36432 total=182159 rate=139.68/s eta=17m 23s
[23:07:04] SurvFace population Grad-CAM | population Grad-CAM extraction | elapsed=6m 11s | progress=30% processed=54648 total=182159 rate=147.38/s eta=14m 25s
[23:08:53] SurvFace population Grad-CAM | population Grad-CAM extraction | elapsed=8m 00s | progress=40% processed=72864 total=182159 rate=151.70/s eta=12m 00s
[23:10:43] SurvFace population Grad-CAM | population Grad-CAM extraction | elapsed=9m 50s | progress=50% processed=91080 total=182159 rate=154.41/s eta=9m 50s
[23:12:32] SurvFace population Grad-CAM | population Grad-CAM extraction | elapsed=11m 39s | progress=60% processed=109296 total=182159 rate=156.28/s eta=7m 46s
[23:14:21] SurvFace population Grad-CAM 

{'run_id': '20260729-R001-cccb59d5',
 'dataset_id': 'survface',
 'population_rows': 463341,
 'gradcam_selected': 182159,
 'heatmaps': 182159,
 'saliency_selection_sha256': '4e5287e7e114d40f17f8e84a5dc1659e0879691332a6d71b562510fb1ca8fe65',
 'region_mask_uid': 'landmark-regions-0cbfdffef6faac5e73801661',
 'next_stage': '03_saliency_feature_validation'}

## 다음 단계

다음은 `01_saliency_feature_validation.ipynb`입니다.

커널을 재시작한 뒤 다음 노트북을 위에서 아래로 실행합니다.